# Experimento de 140 combinaciones, parte 2: clasificación

Resultados de las 108 combinaciones aplicables de clasificación (7 modelos × 4 balanceos × 4 optimizadores, menos KNN con `class_weight`).
Todas las métricas salen de los 5 pliegues externos por bloques espaciales, que no participaron en la elección de hiperparámetros.
El diseño está en `Experimento_1_Diseno.ipynb`.

Para mostrar el mejor caso de cada modelo se elige la combinación de balanceo y optimizador con mejor puntaje interno. Elegirla con la
prueba externa y reportar esa misma prueba volvería a meter el sesgo de selección que la validación anidada quita (Cawley y Talbot, 2010).

## Flujo del experimento

![Flujo del experimento](fig_pipeline_experimento.png)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from src import analisis, config, evaluate, models

maestra, res = analisis.cargar()
clf = res[res.tarea == "clf"]
b = analisis.base()
y = b.y_clf
ORDEN = [m for m in models.MODELOS["clf"] if m in set(clf.modelo)]
print(f"Combinaciones completas de clasificación: {len(clf)} de 108 | modelos: {ORDEN}")

## 1. Las combinaciones, una por una

F1 macro externo medio (y su desviación estándar entre los 5 pliegues). El F1 macro promedia el F1 de las tres clases y es la métrica con la
que se eligen los hiperparámetros: la clase Baja es el 3 % y la exactitud premiaría ignorarla.

In [ ]:
def tabla(metrica):
    m = clf.pivot_table(index=["modelo", "balanceo"], columns="optimizador", values=metrica)
    d = clf.pivot_table(index=["modelo", "balanceo"], columns="optimizador", values=metrica + "_de")
    t = m.round(3).astype(str) + " ± " + d.round(3).astype(str)
    return t.reindex(ORDEN, level=0)[[c for c in ["grilla", "aleatoria", "bayesiana", "genetica"] if c in t.columns]]

tabla("f1_macro")

In [ ]:
tabla("auc_ovr")

## 2. El mejor caso de cada modelo

Métricas externas de la combinación elegida por puntaje interno. Media ± desviación entre pliegues.

In [ ]:
mejores = analisis.mejores_por_modelo(res, "clf").reindex(ORDEN)
cols = ["exactitud", "precision_macro", "recall_macro", "f1_macro", "auc_ovr", "kappa_cuad", "exactitud_bal", "f1_baja", "recall_baja", "brier", "ece"]
t = mejores[["balanceo", "optimizador", "puntaje_interno"]].copy()
for c in cols:
    t[c] = mejores[c].round(3).astype(str) + " ± " + mejores[c + "_de"].round(3).astype(str)
t

## 3. Matriz de confusión de cada modelo

Con las predicciones fuera de pliegue: cada planta aparece una vez, predicha por el modelo que no la vio. Filas = clase real, normalizado por fila
(la diagonal es el recall de cada clase).

In [ ]:
n = len(ORDEN)
fig, axes = plt.subplots(2, 4, figsize=(20, 9.5))
for ax, m in zip(axes.ravel(), ORDEN):
    o = analisis.oof(mejores.loc[m])
    ConfusionMatrixDisplay(confusion_matrix(y, o["pred"].astype(int), normalize="true"), display_labels=config.CLASES).plot(
        ax=ax, values_format=".2f", colorbar=False, cmap="Reds")
    ax.set_title(f"{m}\n{mejores.loc[m, 'balanceo']}, {mejores.loc[m, 'optimizador']}", fontsize=10)
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.tight_layout(); fig.savefig("fig_exp_confusion.png", dpi=130, bbox_inches="tight"); plt.show()

## 4. Curva ROC de cada modelo

Uno contra el resto, una curva por clase. El AUC mide qué tan bien ordenan las probabilidades a cada clase frente a las otras, sin depender
de un umbral.

In [ ]:
Y = label_binarize(y, classes=[0, 1, 2])
fig, axes = plt.subplots(2, 4, figsize=(20, 9.5))
for ax, m in zip(axes.ravel(), ORDEN):
    p = analisis.oof(mejores.loc[m])["proba"]
    for c, nombre in enumerate(config.CLASES):
        fpr, tpr, _ = roc_curve(Y[:, c], p[:, c])
        ax.plot(fpr, tpr, label=f"{nombre} ({roc_auc_score(Y[:, c], p[:, c]):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=.7); ax.set_title(m, fontsize=10); ax.legend(fontsize=8, title="AUC", title_fontsize=8)
    ax.set_xlabel("falsos positivos"); ax.set_ylabel("verdaderos positivos")
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.tight_layout(); fig.savefig("fig_exp_roc.png", dpi=130, bbox_inches="tight"); plt.show()

## 5. Efecto del balanceo

Promedio sobre los cuatro optimizadores. La pregunta es si crear puntos sintéticos (SMOTE, ADASYN) o pesar las clases (`class_weight`)
ayuda a la clase Baja sin dañar al resto, y qué le hace a la calibración de las probabilidades.

In [ ]:
bal = clf.groupby(["modelo", "balanceo"])[["f1_macro", "recall_baja", "f1_baja", "exactitud", "ece"]].mean()
display(bal.unstack("balanceo").reindex(ORDEN).round(3))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, met, titulo in zip(axes, ["f1_macro", "recall_baja", "ece"], ["F1 macro", "Recall de la clase Baja", "ECE (menor es mejor)"]):
    bal[met].unstack("balanceo").reindex(ORDEN).plot.bar(ax=ax, width=.8)
    ax.set_title(titulo); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
fig.tight_layout(); fig.savefig("fig_exp_balanceo.png", dpi=130, bbox_inches="tight"); plt.show()

## 6. Calibración

Un modelo está calibrado si, de las plantas a las que les da 80 % de probabilidad, acierta en más o menos el 80 %. Importa porque las
probabilidades se usan para decidir dónde invertir.

El Brier es el error cuadrático medio de las probabilidades y el ECE, la diferencia media entre confianza y exactitud por grupos de confianza (Guo et al., 2017). En los dos, menor es mejor.

Recalibración: Platt (una sigmoide; Platt, 1999) e isotónica (una función monótona por tramos; Zadrozny y Elkan, 2002). Se ajustan sin fuga:
el calibrador de cada pliegue externo se entrena con las predicciones fuera de pliegue de los otros cuatro.

In [ ]:
filas, curvas = [], {}
pl = None
for m in ORDEN:
    o = analisis.oof(mejores.loc[m])
    p, pl = o["proba"], o["pliegue"]
    for nombre, q in [("original", p), ("Platt", analisis.recalibrar_cruzado(p, y, pl, "platt")),
                      ("isotónica", analisis.recalibrar_cruzado(p, y, pl, "isotonica"))]:
        filas.append({"modelo": m, "probabilidades": nombre, "Brier": evaluate.brier_multiclase(y, q), "ECE": evaluate.ece(y, q),
                      "F1 macro (argmax)": evaluate.metricas_clf(y, q.argmax(1), None)["f1_macro"]})
        curvas[(m, nombre)] = analisis.confiabilidad(y, q)
cal = pd.DataFrame(filas).set_index(["modelo", "probabilidades"])
cal.round(4)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9.5))
for ax, m in zip(axes.ravel(), ORDEN):
    for nombre, estilo in [("original", "-o"), ("Platt", "-s"), ("isotónica", "-^")]:
        x, f = curvas[(m, nombre)]
        ax.plot(x, f, estilo, ms=4, label=f"{nombre} (ECE {cal.loc[(m, nombre), 'ECE']:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=.7); ax.set_xlim(.3, 1); ax.set_ylim(.3, 1)
    ax.set_title(m, fontsize=10); ax.set_xlabel("confianza media"); ax.set_ylabel("exactitud"); ax.legend(fontsize=8)
for ax in axes.ravel()[len(ORDEN):]:
    ax.axis("off")
fig.tight_layout(); fig.savefig("fig_exp_calibracion.png", dpi=130, bbox_inches="tight"); plt.show()

## 7. Estabilidad de los hiperparámetros elegidos

Si cada pliegue externo elige valores muy distintos, el modelo es sensible a qué bloques le tocan, o el puntaje es plano en esa región
del espacio y da igual el valor.

In [ ]:
mejor = mejores.puntaje_interno.idxmax()
print("Modelo con mejor puntaje interno:", mejor, "|", mejores.loc[mejor, "balanceo"], "|", mejores.loc[mejor, "optimizador"])
analisis.params_por_pliegue(maestra, mejores.loc[mejor]).round(4)

## 8. Interpretación

Los tres modelos de árboles quedan arriba: Random Forest (F1 macro 0.894), XGBoost (0.888) y el árbol de decisión (0.881), con AUC de 0.97 o más. Les siguen el SVM RBF (0.814) y KNN (0.799). La regresión logística (0.629) y Naive Bayes (0.580) quedan muy por debajo. Dentro de un mismo modelo el optimizador casi no cambia el resultado: en Random Forest las cuatro búsquedas dan entre 0.893 y 0.894.

El balanceo no mejoró el F1 macro. Sin balanceo sale el mejor valor en KNN, Naive Bayes, el árbol, Random Forest, XGBoost y el SVM; la excepción es la regresión logística, que pasa de 0.600 a 0.629 con `class_weight` o SMOTE. Las tres técnicas sí suben el recall de la clase Baja (en Random Forest, de 0.860 a entre 0.915 y 0.924), pero con más falsos positivos, así que el F1 de Baja baja un poco (0.860 sin balanceo y entre 0.846 y 0.848 con balanceo). También empeoran la calibración: el ECE de Random Forest sube de 0.016 a entre 0.027 y 0.033.

Los árboles salen bien calibrados sin tocarlos (ECE de 0.012 a 0.015 con las predicciones fuera de pliegue). La recalibración isotónica los baja a entre 0.003 y 0.005, y la de Platt los empeora. En Naive Bayes y en la regresión logística la isotónica reduce el ECE (de 0.072 a 0.019 y de 0.055 a 0.027), pero en Naive Bayes el F1 cae de 0.583 a 0.306: sirve para mejorar las probabilidades y no para decidir la clase.

Los hiperparámetros de Random Forest cambian bastante entre pliegues (de 96 a 186 árboles y de 2 a 16 muestras por hoja) sin que cambie el puntaje, así que el puntaje es casi plano cerca del óptimo.

## Referencias

- Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079-2107.
- Chawla, N. V., Bowyer, K. W., Hall, L. O. y Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *Journal of Artificial Intelligence Research, 16*, 321-357. https://doi.org/10.1613/jair.953
- Guo, C., Pleiss, G., Sun, Y. y Weinberger, K. Q. (2017). On calibration of modern neural networks. En *Proceedings of ICML* (PMLR 70, pp. 1321-1330).
- He, H., Bai, Y., Garcia, E. A. y Li, S. (2008). ADASYN: Adaptive synthetic sampling approach for imbalanced learning. En *2008 IEEE IJCNN* (pp. 1322-1328). https://doi.org/10.1109/IJCNN.2008.4633969
- Platt, J. (1999). Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. En *Advances in Large Margin Classifiers* (pp. 61-74). MIT Press.
- Zadrozny, B. y Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. En *Proceedings of the 8th ACM SIGKDD* (pp. 694-699). https://doi.org/10.1145/775047.775151